In [1]:
%pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 49.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

file_id = '10FwLy34tDMePO75ldYlM4RDoATsLkPfh'   # new file
url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

loan_df = pd.read_csv(url)
loan_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 391166 entries, 0 to 391165
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   loan_amnt                   391166 non-null  int64  
 1   term                        391166 non-null  int64  
 2   int_rate                    391166 non-null  float64
 3   installment                 391166 non-null  float64
 4   sub_grade                   391166 non-null  str    
 5   emp_length                  391166 non-null  float64
 6   home_ownership              391166 non-null  str    
 7   annual_inc                  391166 non-null  float64
 8   verification_status         391166 non-null  str    
 9   purpose                     391166 non-null  str    
 10  dti                         391166 non-null  float64
 11  delinq_2yrs                 391166 non-null  float64
 12  inq_last_6mths              391166 non-null  float64
 13  open_acc                 

In [5]:
import numpy as np

# Copy dataset to avoid overwriting original
df_policy = loan_df.copy()

# Replace 0s to avoid division errors
df_policy["annual_inc"] = df_policy["annual_inc"].replace(0, np.nan)
df_policy["loan_amnt"] = df_policy["loan_amnt"].replace(0, np.nan)
df_policy["installment"] = df_policy["installment"].replace(0, np.nan)

# -----------------------------
#  Financial Ratios
# -----------------------------

# Income relative to loan size -> higher = more ability to repay
df_policy["income_to_loan_ratio"] = df_policy["annual_inc"] / df_policy["loan_amnt"]

# Loan size relative to income -> higher = more risk / burden
df_policy["loan_to_income_ratio"] = df_policy["loan_amnt"] / df_policy["annual_inc"]

# Annual payment burden relative to income -> measures affordability
df_policy["installment_to_income_ratio"] = (df_policy["installment"] * 12) / df_policy["annual_inc"]

# Monthly income vs payment -> ability to comfortably make payments
df_policy["income_to_installment_ratio"] = (df_policy["annual_inc"] / 12) / df_policy["installment"]

# Existing revolving debt burden -> captures current financial strain
df_policy["revol_bal_to_income_ratio"] = df_policy["revol_bal"] / df_policy["annual_inc"]

# Available credit vs total credit -> higher = more unused capacity
df_policy["available_credit_ratio"] = df_policy["bc_open_to_buy"] / df_policy["total_rev_hi_lim"]

# Remaining utilization capacity -> higher = lower current credit usage
df_policy["all_util_headroom"] = 100 - df_policy["all_util"]

# -----------------------------
# Clean up infinite values
# -----------------------------
df_policy.replace([np.inf, -np.inf], np.nan, inplace=True)

# Quick check
ratio_cols = [
    "income_to_loan_ratio",
    "loan_to_income_ratio",
    "installment_to_income_ratio",
    "income_to_installment_ratio",
    "revol_bal_to_income_ratio",
    "available_credit_ratio",
    "all_util_headroom"
]

df_policy[ratio_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
income_to_loan_ratio,391166.0,7.220893,7.643163,0.001219,3.437500,5.000000,8.000000,489.795918
loan_to_income_ratio,391166.0,0.218623,1.381245,0.002042,0.125000,0.200000,0.290909,820.512821
installment_to_income_ratio,391166.0,0.080646,0.479542,0.000814,0.046336,0.072276,0.105364,282.569231
income_to_installment_ratio,391166.0,19.112476,19.195519,0.003539,9.490899,13.835851,21.581322,1228.803146
revol_bal_to_income_ratio,391166.0,0.228782,1.289628,0.000000,0.098340,0.179552,0.295727,793.641026
available_credit_ratio,366772.0,0.274964,0.229596,0.000000,0.082452,0.218924,0.420906,1.000000
all_util_headroom,149582.0,41.784372,20.965805,-98.000000,27.000000,40.000000,56.000000,100.000000
